# Laboratorio 02: Representación de Problemas y Búsqueda en Amplitud (BFS)

## 4.2. Representación de un problema mediante estados

### Parte A: Ficha del problema del repartidor
caso: un repartidor debe ir desde el
Almacén hasta la Zona Norte, pudiendo pasar por distintos puntos intermedios de la ciudad.

| Componente del Problema | Descripción |
| :--- | :--- |
| **Estado inicial** | Ubicación inicial del repartidor: `"Almacen"`. |
| **Estado objetivo** | Punto de llegada deseado: `"ZonaNorte"`. |
| **Espacio de estados** | Todos los puntos alcanzables en la ciudad: `{"Almacen", "Centro", "Terminal", "Mercado", "ZonaNorte"}`. |
| **Acciones posibles** | Desplazarse hacia un punto adyacente conectado directamente por una vía. |
| **Costo de cada acción** | Costo unitario ($c = 1$) por cada tramo recorrido entre dos puntos. |

### Parte B: Representar el mapa de la ciudad como un grafo en Python, usando un diccionario de listas (cada clave es un punto, cada valor es la lista de puntos conectados directamente):

In [3]:
# Parte B: Representación del mapa como un grafo en Python
# Se utiliza un diccionario donde cada Clave (Key) es un punto/nodo, 
# y su Valor (Value) es la lista de puntos conectados directamente (vecinos).

mapa = {
    "Almacen": ["Centro", "Terminal"],
    "Centro": ["Almacen", "ZonaNorte", "Mercado"],
    "Terminal": ["Almacen", "Mercado"],
    "Mercado": ["Centro", "Terminal", "ZonaNorte"],
    "ZonaNorte": ["Centro", "Mercado"],
}

print("=== REPRESENTACIÓN DEL GRAFO (MAPA DE LA CIUDAD) ===")
for punto, vecinos in mapa.items():
    print(f"Desde '{punto}' se puede ir a: {vecinos}")

=== REPRESENTACIÓN DEL GRAFO (MAPA DE LA CIUDAD) ===
Desde 'Almacen' se puede ir a: ['Centro', 'Terminal']
Desde 'Centro' se puede ir a: ['Almacen', 'ZonaNorte', 'Mercado']
Desde 'Terminal' se puede ir a: ['Almacen', 'Mercado']
Desde 'Mercado' se puede ir a: ['Centro', 'Terminal', 'ZonaNorte']
Desde 'ZonaNorte' se puede ir a: ['Centro', 'Mercado']


## 4.3. Implementación y Rastreo de Búsqueda en Amplitud (BFS)

### ¿Cómo funciona BFS (Breadth-First Search)?
1. **Estructura tipo Cola (FIFO):** Explora el grafo nivel por nivel (primero todos los vecinos directos, luego los vecinos de estos). Usa `cola.pop(0)` para sacar siempre el elemento más antiguo.
2. **Garantía de Optimizabilidad:** Al explorar por niveles concéntricos, garantiza encontrar la ruta con el **menor número de tramos (pasos)** en grafos sin pesos.
3. **Control de Ciclos:** Mantiene un conjunto de `visitados` para evitar entrar en bucles infinitos entre nodos interconectados.

### Tabla de Rastreo de BFS (Ejecución Paso a Paso)

| Iteración | Cola de Caminos (FIFO) | Camino Extraído (`pop(0)`) | Nodo Actual | ¿Es Objetivo? (`ZonaNorte`) | ¿Visitado? | Acción / Nuevos Caminos Agregados a la Cola |
| :---: | :--- | :--- | :---: | :---: | :---: | :--- |
| **0** | `[['Almacen']]` | `['Almacen']` | `Almacen` | No | No | Marca `Almacen`. Agrega: `['Almacen', 'Centro']`, `['Almacen', 'Terminal']` |
| **1** | `[['Almacen', 'Centro']], [['Almacen', 'Terminal']]` | `['Almacen', 'Centro']` | `Centro` | No | No | Marca `Centro`. Agrega: `['Almacen', 'Centro', 'Almacen']`, `['Almacen', 'Centro', 'ZonaNorte']`, `['Almacen', 'Centro', 'Mercado']` |
| **2** | `[['Almacen', 'Terminal']], ...` | `['Almacen', 'Terminal']` | `Terminal` | No | No | Marca `Terminal`. Agrega vecinos de Terminal a la cola. |
| **3** | `[['Almacen', 'Centro', 'Almacen']], ...` | `['Almacen', 'Centro', 'Almacen']` | `Almacen` | No | **Sí** | Se descarta porque `Almacen` ya fue visitado previamente. |
| **4** | `[['Almacen', 'Centro', 'ZonaNorte']], ...` | `['Almacen', 'Centro', 'ZonaNorte']` | `ZonaNorte` | **¡SÍ!** | - | **¡Objetivo alcanzado!** Retorna el camino: `['Almacen', 'Centro', 'ZonaNorte']`. |

In [4]:
def bfs(grafo, inicio, objetivo):
    """
    Busca el camino con menos pasos entre 'inicio' y 'objetivo' usando BFS.
    
    Parámetros:
        grafo (dict): Diccionario de listas que representa la estructura del mapa.
        inicio (str): Nodo inicial desde donde parte el repartidor.
        objetivo (str): Nodo destino al que se quiere llegar.
    """
    # 1. Inicializamos la cola guardando CAMINOS COMPLETOS (listas), no solo nodos.
    #    Empezamos con un camino que contiene únicamente el estado inicial.
    cola = [[inicio]] 
    
    # 2. Conjunto (set) para registrar los nodos ya procesados y evitar ciclos.
    visitados = set()
    
    # 3. Mientas la cola no esté vacía, seguimos buscando.
    while cola:
        # Extraemos el PRIMER camino de la cola (Comportamiento FIFO: Primero en Entrar, Primero en Salir)
        camino = cola.pop(0) 
        
        # El nodo actual en el que estamos parados es el ÚLTIMO elemento del camino extraído
        nodo_actual = camino[-1]
        
        # EVALUACIÓN DEL OBJETIVO: ¿Llegamos al destino?
        if nodo_actual == objetivo:
            return camino  # Retornamos la secuencia completa de pasos
        
        # SI NO HA SIDO VISITADO: Procesamos sus vecinos
        if nodo_actual not in visitados:
            visitados.add(nodo_actual)  # Marcamos el nodo como visitado
            
            # Recorremos todos los nodos conectados directamente con el nodo actual
            for vecino in grafo[nodo_actual]:
                # CONSTRUCCIÓN DEL NUEVO CAMINO:
                # Concatenamos el camino actual con el nuevo vecino (camino + [vecino])
                nuevo_camino = camino + [vecino]
                
                # Agregamos el nuevo camino al final de la cola
                cola.append(nuevo_camino)
                
    return None  # Si la cola se vacía y no encontramos el objetivo, no hay ruta posible

# PRUEBA DE LA FUNCIÓN
camino_resultado = bfs(mapa, "Almacen", "ZonaNorte")

print("=== RESULTADO DE LA BÚSQUEDA BFS ===")
print("Ruta óptima encontrada:", camino_resultado)
print(f"Número total de tramos/pasos: {len(camino_resultado) - 1}")

=== RESULTADO DE LA BÚSQUEDA BFS ===
Ruta óptima encontrada: ['Almacen', 'Centro', 'ZonaNorte']
Número total de tramos/pasos: 2


## 4.4. Vectores y matrices con NumPy

En IA, las observaciones del mundo real se representan mediante **vectores de características**, los cuales se agrupan verticalmente para formar **matrices de datos** (datasets).

### Parte A: Creación de vectores y la matriz dataset
Representamos cuatro observaciones de sensores (cada una con `[temperatura, humedad]`) como vectores individuales y los agrupamos en una matriz de 2 dimensiones:
* **Filas (Eje 0):** Representan cada observación individual ($obs_1, obs_2, obs_3, obs_4$).
* **Columnas (Eje 1):** Representan las variables medidas (Columna 0 = Temperatura, Columna 1 = Humedad).

In [1]:
import numpy as np

# Definición de cada observación como un vector de 1D: [temperatura, humedad]
obs1 = np.array([22.5, 60])  # Observación 1
obs2 = np.array([19.0, 45])  # Observación 2
obs3 = np.array([25.3, 70])  # Observación 3
obs4 = np.array([21.0, 50])  # Observación 4

# Unimos los vectores para construir la matriz bidimensional (Dataset)
dataset = np.array([obs1, obs2, obs3, obs4])

# Mostramos la matriz completa en pantalla
print("=== MATRIZ DATASET DE SENSORES ===")
print(dataset)

# Muestra las dimensiones en formato (filas, columnas)
print("\nForma de la matriz (filas, columnas):", dataset.shape)

=== MATRIZ DATASET DE SENSORES ===
[[22.5 60. ]
 [19.  45. ]
 [25.3 70. ]
 [21.  50. ]]

Forma de la matriz (filas, columnas): (4, 2)


### Parte B: Acceso e indexación de filas y columnas

Para extraer información específica de una matriz en NumPy utilizamos la notación `[fila, columna]`:
* `dataset[i]`: Accede a la fila completa del índice $i$.
* `dataset[:, j]`: El operador `:` indica "todas las filas", permitiendo extraer únicamente la columna en el índice $j$.

In [2]:
# Accedemos a la primera fila completa (índice 0 -> obs1)
print("Primera observacion (fila 0):", dataset[0])

# Accedemos a la primera columna completa (índice 0 -> todas las temperaturas)
print("Todas las temperaturas (columna 0):", dataset[:, 0])

# Accedemos a la segunda columna completa (índice 1 -> todas las humedades)
print("Todas las humedades (columna 1):", dataset[:, 1])

Primera observacion (fila 0): [22.5 60. ]
Todas las temperaturas (columna 0): [22.5 19.  25.3 21. ]
Todas las humedades (columna 1): [60. 45. 70. 50.]


## 4.5. Producto escalar y distancia euclidiana

### Parte A: Producto escalar entre observaciones

El **producto escalar** (o producto punto) de dos vectores suma la multiplicación de sus componentes homólogas:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i \cdot b_i = (a_1 \cdot b_1) + (a_2 \cdot b_2)$$

Para $obs_1 = [22.5, 60]$ y $obs_2 = [19.0, 45]$:
$$\text{Producto} = (22.5 \times 19.0) + (60 \times 45) = 427.5 + 2700 = 3127.5$$

In [3]:
# Cálculo del producto escalar entre obs1 y obs2 usando la función np.dot
producto = np.dot(obs1, obs2)

print("=== PRODUCTO ESCALAR ===")
print("Producto escalar entre obs1 y obs2:", producto)

=== PRODUCTO ESCALAR ===
Producto escalar entre obs1 y obs2: 3127.5


### Parte B: Distancia euclidiana (Fórmula manual vs NumPy)

La **distancia euclidiana** mide la separación geométrica en línea recta entre dos observaciones. Es la métrica fundamental usada en algoritmos como **KNN** para medir similitud:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Compararemos dos métodos:
1. **Manual:** Aplicando restas, cuadrados, suma y raíz cuadrada explícitamente.
2. **NumPy:** Usando la función nativa `np.linalg.norm()` (Norma $L_2$).

In [4]:
# 1. Cálculo mediante la fórmula matemática manual paso a paso:
# Paso 1: Resta vectorial (obs1 - obs2)
# Paso 2: Elevar al cuadrado (** 2)
# Paso 3: Sumar las componentes (np.sum)
# Paso 4: Raíz cuadrada (np.sqrt)
distancia_manual = np.sqrt(np.sum((obs1 - obs2) ** 2))
print("Distancia manual:", distancia_manual)

# 2. Cálculo directo optimizado mediante la función de NumPy
distancia_numpy = np.linalg.norm(obs1 - obs2)
print("Distancia con NumPy:", distancia_numpy)

# Verificación de igualdad entre ambos métodos
print("\n¿Ambos resultados son numéricamente idénticos?:", distancia_manual == distancia_numpy)

Distancia manual: 15.402921800749363
Distancia con NumPy: 15.402921800749363

¿Ambos resultados son numéricamente idénticos?: True


# 5. Ejercicio Práctico de Laboratorio

## 5.1. Planteamiento del problema
En esta sección práctica resolvemos dos tareas fundamentales:
1. **Búsqueda en Profundidad (DFS):** Implementar la búsqueda en profundidad sobre el mapa de la ciudad usando una estructura de Pila (LIFO) para comparar su comportamiento y ruta contra el algoritmo BFS de la Sección 4.3.
2. **Observación más cercana (Fundamento de KNN):** Calcular la distancia euclidiana entre un conjunto de observaciones conocidas y un nuevo dato para identificar cuál es el más parecido.

---

## 5.2. Parte A: Implementación de DFS

A diferencia de BFS (que explora por niveles concéntricos usando una Cola FIFO con `pop(0)`), el algoritmo **DFS (Depth-First Search)** explora adentrándose lo más profundo posible en cada rama antes de retroceder, utilizando una **Pila LIFO** (*Last In, First Out*) mediante `.pop()`.

In [7]:
import numpy as np

# Definición del mapa de la ciudad (se incluye para garantizar que no lance NameError)
mapa = {
    "Almacen": ["Centro", "Terminal"],
    "Centro": ["Almacen", "ZonaNorte", "Mercado"],
    "Terminal": ["Almacen", "Mercado"],
    "Mercado": ["Centro", "Terminal", "ZonaNorte"],
    "ZonaNorte": ["Centro", "Mercado"],
}

def dfs(grafo, inicio, objetivo):
    """
    Busca un camino entre 'inicio' y 'objetivo' usando DFS (Pila LIFO).
    """
    pila = [[inicio]]  # Inicializamos la pila con el camino inicial
    visitados = set()  # Conjunto para registrar nodos ya procesados
    
    while pila:
        camino = pila.pop()  # Extrae el ÚLTIMO camino de la pila (LIFO)
        nodo_actual = camino[-1]  # El nodo actual es la última posición del camino
        
        # Evaluación de la condición objetivo
        if nodo_actual == objetivo:
            return camino
        
        # Si el nodo no ha sido visitado previamente
        if nodo_actual not in visitados:
            visitados.add(nodo_actual)  # Marcamos el nodo como visitado
            
            for vecino in grafo[nodo_actual]:
                # COMPLETADO: Construimos el nuevo camino extendiéndolo con el vecino
                nuevo_camino = camino + [vecino]
                
                # COMPLETADO: Agregamos el nuevo camino a la pila
                pila.append(nuevo_camino)
                
    return None  # Retorna None si no existe ruta

# Ejecución del algoritmo DFS
resultado_dfs = dfs(mapa, "Almacen", "ZonaNorte")

print("=== RESULTADO DE BÚSQUEDA DFS ===")
print("Camino encontrado (DFS):", resultado_dfs)

=== RESULTADO DE BÚSQUEDA DFS ===
Camino encontrado (DFS): ['Almacen', 'Terminal', 'Mercado', 'ZonaNorte']


### Comparación de Resultados: BFS vs. DFS

* **Camino encontrado por BFS (Sección 4.3):** `['Almacen', 'Centro', 'ZonaNorte']` (Total: 2 tramos).
* **Camino encontrado por DFS (Sección 5.2):** `['Almacen', 'Terminal', 'Mercado', 'ZonaNorte']` (Total: 3 tramos).

**Respuestas a las preguntas de la guía:**
1. **¿Son el mismo camino?:** **No**, son caminos totalmente diferentes.
2. **¿Cuál tiene menos pasos?:** **BFS tiene menos pasos** (2 tramos frente a los 3 tramos de DFS).
3. **Explicación:** BFS explora en anchura (nivel por nivel de cercanía), por lo que siempre garantiza encontrar el camino con la menor cantidad de pasos. Por su parte, DFS saca el último elemento ingresado a la pila (LIFO), lo que provoca que se adentre profundamente por la rama de `Terminal` $\rightarrow$ `Mercado` $\rightarrow$ `ZonaNorte` y devuelva el primer camino que llegue al objetivo, aunque requiera más tramos.

## 5.3. Parte B: Encontrar la observación más cercana

Dado el dataset de cuatro observaciones de temperatura y humedad, calculamos la distancia euclidiana entre una **nueva observación** (`[20.5°C, 48% humedad]`) y cada una de las existentes en el dataset histórico, con el objetivo de identificar cuál es la más similar (principio fundamental del algoritmo KNN).

In [8]:
# Recreamos el dataset previo para evitar errores de ejecución
obs1 = np.array([22.5, 60])
obs2 = np.array([19.0, 45])
obs3 = np.array([25.3, 70])
obs4 = np.array([21.0, 50])
dataset = np.array([obs1, obs2, obs3, obs4])

# Nueva observación a clasificar
nueva_observacion = np.array([20.5, 48])
distancias = []

print("=== DISTANCIAS A CADA OBSERVACIÓN DEL DATASET ===")

for i, obs in enumerate(dataset):
    # COMPLETADO: Cálculo de distancia euclidiana entre 'nueva_observacion' y 'obs'
    d = np.linalg.norm(nueva_observacion - obs)
    distancias.append(d)
    print(f"Distancia a la observacion {i} {obs}: {d:.4f}")

# np.argmin nos da el índice del valor más pequeño en la lista de distancias
indice_mas_cercano = np.argmin(distancias)

print("\n=== RESULTADO ===")
print("La observacion mas parecida es la numero", indice_mas_cercano)
print(f"Valores del registro más cercano: {dataset[indice_mas_cercano]}")

=== DISTANCIAS A CADA OBSERVACIÓN DEL DATASET ===
Distancia a la observacion 0 [22.5 60. ]: 12.1655
Distancia a la observacion 1 [19. 45.]: 3.3541
Distancia a la observacion 2 [25.3 70. ]: 22.5175
Distancia a la observacion 3 [21. 50.]: 2.0616

=== RESULTADO ===
La observacion mas parecida es la numero 3
Valores del registro más cercano: [21. 50.]


## 5.4. Parte C: Reflexión

### 1. ¿Por qué la distancia euclidiana es una forma razonable de medir qué tan parecidas son dos observaciones?
La distancia euclidiana es una medida adecuada porque calcula la separación en línea recta dentro de un espacio vectorial geométrico:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Si dos observaciones poseen valores numéricos muy similares en sus características (temperatura y humedad cercanas), la diferencia entre sus componentes $(a_i - b_i)$ será pequeña, produciendo una distancia corta. Por ende, a **menor distancia euclidiana, mayor es el grado de similitud** entre los datos.

---

### 2. ¿En qué casos podría no ser la mejor opción?
Falla cuando las características del dataset se miden en **escalas o magnitudes numéricas muy dispares**.

* **Ejemplo práctico (Soles vs. Años):**
  Si comparamos dos registros con las variables *Sueldo mensual en Soles* y *Edad en Años*:
  * Registro 1: `[S/. 3000, 25 años]`
  * Registro 2: `[S/. 1500, 30 años]`
  
  Al aplicar la fórmula:
  $$\text{Diferencia de Sueldo}^2 = (3000 - 1500)^2 = 2,250,000$$
  $$\text{Diferencia de Edad}^2 = (25 - 30)^2 = 25$$

* **Consecuencia:** La variable del sueldo domina completamente el resultado de la distancia, anulando el efecto de la edad.
* **Solución:** En estos escenarios es obligatorio aplicar **normalización o estandarización** de datos (como *Min-Max Scaling* o *StandardScaler*) antes de calcular la distancia, o utilizar otras métricas como la **distancia de Manhattan** o la **similitud de coseno**.